In [ ]:
month_rto_by_client['first_campaign_dt'] = pd.to_datetime(
    month_rto_by_client['first_campaign_dt']
)

month_rto_by_client['month_dt'] = pd.to_datetime(
    month_rto_by_client['month_dt']
)

month_rto_by_client['month_shift'] = (
    (month_rto_by_client['month_dt'].dt.year - month_rto_by_client['first_campaign_dt'].dt.year) * 12
    + (month_rto_by_client['month_dt'].dt.month - month_rto_by_client['first_campaign_dt'].dt.month)
)

In [ ]:
def iqr_filter(group):
    q1 = group['month_spend'].quantile(0.25)
    q3 = group['month_spend'].quantile(0.75)

    iqr = q3 - q1

    return group[
        group['month_spend'].between(
            q1 - 1.5 * iqr,
            q3 + 1.5 * iqr
        )
    ]


month_rto_by_client_cleaned = (
    month_rto_by_client
    .groupby(['campaigns_cnt', 'month_shift'], group_keys=False)
    .apply(iqr_filter)
    .reset_index(drop=True)
)

In [ ]:
month_rto_by_cohorts = (
    month_rto_by_client_cleaned
    .groupby(['campaigns_cnt', 'month_shift'], as_index=False)
    .agg(
        avg_spend_per_client=('month_spend', 'mean'),
        median_spend_per_client=('month_spend', 'median'),
        client_cnt=('client_id', 'nunique')
    )
    .sort_values(['campaigns_cnt', 'month_shift'])
)

In [ ]:
def plot_rto_by_metric(ax, data, metric, title):
    for cohort in sorted(data['campaigns_cnt'].unique()):
        part = data[data['campaigns_cnt'] == cohort]

        ax.plot(
            part['month_shift'],
            part[metric],
            marker='o',
            linewidth=2,
            label=f'{cohort} камп.'
        )

    ax.axvline(x=0, linestyle='--', alpha=0.5)

    ax.set_title(title)
    ax.set_xlabel('Период относительно первой кампании')
    ax.grid(True, alpha=0.3)

    ax.set_xticks([-1, 0, 1, 2, 3, 4])
    ax.set_xticklabels(['-1 мес', '1-я камп.', '+1', '+2', '+3', '+4'])

    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'.replace(',', ' '))
    )


fig, axes = plt.subplots(1, 2, figsize=(18, 7), sharey=True)

plot_rto_by_metric(
    axes[0],
    month_rto_by_cohorts,
    'median_spend_per_client',
    'Медианный РТО'
)

plot_rto_by_metric(
    axes[1],
    month_rto_by_cohorts,
    'avg_spend_per_client',
    'Средний РТО'
)

axes[0].set_ylabel('РТО')

handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    title='Кол-во кампаний',
    bbox_to_anchor=(1.02, 0.95),
    loc='upper left'
)

fig.suptitle(
    'РТО по когортам относительно первой кампании',
    fontsize=16
)

plt.tight_layout()
plt.show()